# ILLM Lab Assignment 2
## TF-IDF Normalisation for Multilingual Language Identification

**Objective:** Use the same multilingual dataset and feature representation from Assignment 1 and compare six combinations of normalised and unnormalised TF and IDF vectors.

### Required experiments
1. Original TF + Original IDF
2. Total-word normalised TF + Original IDF
3. Maximum-frequency normalised TF + Original IDF
4. Original TF + Normalised IDF
5. Total-word normalised TF + Normalised IDF
6. Maximum-frequency normalised TF + Normalised IDF

All TF, IDF, TF-IDF, multiclass Logistic Regression, and Macro-F1 calculations are implemented without scikit-learn.

In [1]:
# Cell 1: Import the libraries used for the assignment.
# Only basic Python, NumPy, and Hugging Face datasets are used.
# No scikit-learn is used.

import math
import random
import time
import numpy as np

print("NumPy version:", np.__version__)
print("Imports successful.")

NumPy version: 2.5.1
Imports successful.


## 1. Load the same IndicCorpV2 dataset used in Assignment 1

The corpus is loaded in **streaming mode** so the complete corpus is not downloaded to disk.

In [2]:
# Cell 2: Load IndicCorpV2 in streaming mode.
# Streaming prevents the complete corpus from being downloaded/prepared locally.

from datasets import load_dataset

dataset = load_dataset(
    "ai4bharat/IndicCorpV2",
    "indiccorp_v2",
    streaming=True
)

languages = list(dataset.keys())

print("Number of languages:", len(languages))
print(languages)

c:\Users\mdevr\Desktop\ILLM lab\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Number of languages: 23
['asm_Beng', 'ben_Beng', 'brx_Deva', 'doi_Deva', 'gom_Deva', 'guj_Gujr', 'hin_Deva', 'kan_Knda', 'kas_Arab', 'mai_Deva', 'mal_Mlym', 'mar_Deva', 'mni_Mtei', 'npi_Deva', 'ory_Orya', 'pan_Guru', 'san_Deva', 'snd_Deva', 'tam_Taml', 'tel_Telu', 'urd_Arab', 'khasi', 'santhali']


## 2. Recreate the 23,000-sentence dataset

We reproduce the Assignment 1 setup: **1000 sentences per language for 23 languages**.

In [3]:
# Cell 3: Define a sentence splitter without NLTK or other tokenisation libraries.

def split_into_sentences(text):
    delimiters = ['.', '?', '!', '।']
    sentences = []
    current = ""

    for ch in text:
        current += ch

        if ch in delimiters:
            sentence = current.strip()
            if sentence:
                sentences.append(sentence)
            current = ""

    if current.strip():
        sentences.append(current.strip())

    return sentences

In [4]:
# Cell 4: Collect 1000 sentences for one language.

def collect_sentences(language, limit=1000):
    data = []
    language_dataset = dataset[language]

    for sample in language_dataset:
        text = sample["text"]
        sentences = split_into_sentences(text)

        for sentence in sentences:
            sentence = sentence.strip()

            if not sentence:
                continue

            data.append((sentence, language))

            if len(data) == limit:
                return data

    return data

In [5]:
# Cell 5: Recreate the complete 23-language dataset.

all_data = []

for language in languages:
    print("Processing:", language)

    language_data = collect_sentences(language, 1000)

    print("Collected:", len(language_data))
    all_data.extend(language_data)

print()
print("Total sentences:", len(all_data))

Processing: asm_Beng
Collected: 1000
Processing: ben_Beng
Collected: 1000
Processing: brx_Deva
Collected: 1000
Processing: doi_Deva
Collected: 1000
Processing: gom_Deva
Collected: 1000
Processing: guj_Gujr
Collected: 1000
Processing: hin_Deva
Collected: 1000
Processing: kan_Knda
Collected: 1000
Processing: kas_Arab
Collected: 1000
Processing: mai_Deva
Collected: 1000
Processing: mal_Mlym
Collected: 1000
Processing: mar_Deva
Collected: 1000
Processing: mni_Mtei
Collected: 1000
Processing: npi_Deva
Collected: 1000
Processing: ory_Orya
Collected: 1000
Processing: pan_Guru
Collected: 1000
Processing: san_Deva
Collected: 1000
Processing: snd_Deva
Collected: 1000
Processing: tam_Taml
Collected: 1000
Processing: tel_Telu
Collected: 1000
Processing: urd_Arab
Collected: 1000
Processing: khasi
Collected: 1000
Processing: santhali
Collected: 1000

Total sentences: 23000


## 3. Stratified 80:10:10 train/validation/test split

In [6]:
# Cell 6: Create the same stratified 80:10:10 split used in Assignment 1.
# A fixed seed makes the split reproducible.

random.seed(42)

language_groups = {}

for sentence, label in all_data:
    if label not in language_groups:
        language_groups[label] = []

    language_groups[label].append((sentence, label))

train = []
validation = []
test = []

for label in language_groups:
    samples = language_groups[label]
    random.shuffle(samples)

    train.extend(samples[:800])
    validation.extend(samples[800:900])
    test.extend(samples[900:1000])

print("Training:", len(train))
print("Validation:", len(validation))
print("Test:", len(test))

Training: 18400
Validation: 2300
Test: 2300


## 4. Feature extraction

The feature representation is kept the same as Assignment 1:
- Word unigrams
- Word bigrams
- Character 2-grams
- Character 3-grams
- Character 4-grams

In [7]:
# Cell 7: Extract the same word and character n-gram features as Assignment 1.

def extract_features(sentence):
    features = []

    # Word unigrams
    words = sentence.split()

    for word in words:
        features.append(("WU", word))

    # Word bigrams
    for i in range(len(words) - 1):
        features.append(("WB", words[i] + " " + words[i + 1]))

    # Character n-grams
    text = sentence.replace(" ", "")

    for i in range(len(text) - 1):
        features.append(("C2", text[i:i+2]))

    for i in range(len(text) - 2):
        features.append(("C3", text[i:i+3]))

    for i in range(len(text) - 3):
        features.append(("C4", text[i:i+4]))

    return features

## 5. Build the shared vocabulary and document statistics

The vocabulary and document frequencies are computed **only from the training set** and are reused by all six experiments.

In [8]:
# Cell 8: Build the vocabulary from training data only.

vocabulary = {}
feature_index = 0

for sentence, label in train:
    for feature in extract_features(sentence):
        if feature not in vocabulary:
            vocabulary[feature] = feature_index
            feature_index += 1

print("Vocabulary size:", len(vocabulary))

Vocabulary size: 1084767


In [9]:
# Cell 9: Compute document frequency for every feature.

document_frequency = {}

for sentence, label in train:
    unique_features = set(extract_features(sentence))

    for feature in unique_features:
        document_frequency[feature] = document_frequency.get(feature, 0) + 1

print("Number of features with DF:", len(document_frequency))

Number of features with DF: 1084767


## 6. IDF definitions

### Original IDF
\[
IDF(t)=\log\left(\frac{N+1}{DF(t)+1}\right)+1
\]

### Normalised IDF
For this assignment, the IDF is normalised relative to the document frequency of the most frequent feature in the training corpus. We use:

\[
IDF_{norm}(t)=\frac{\log\left(\frac{N}{DF(t)}\right)}
{\log\left(\frac{N}{DF_{max}}\right)}
\]

When the denominator is zero, the normalised IDF is set to zero.

Here, \(DF_{max}\) is the largest document frequency observed in the training corpus.

In [10]:
# Cell 10: Compute original and normalised IDF values.

N = len(train)

idf_original = {}
idf_normalized = {}

max_df = max(document_frequency.values())

for feature, df in document_frequency.items():
    # Original IDF used in Assignment 1
    idf_original[feature] = math.log((N + 1) / (df + 1)) + 1

    # Normalised IDF
    numerator = math.log(N / df)

    denominator = math.log(N / max_df)

    if denominator == 0:
        idf_normalized[feature] = 0.0
    else:
        idf_normalized[feature] = numerator / denominator

print("Maximum document frequency:", max_df)
print("Original IDF values:", len(idf_original))
print("Normalised IDF values:", len(idf_normalized))

Maximum document frequency: 2368
Original IDF values: 1084767
Normalised IDF values: 1084767


## 7. TF definitions

We implement the three TF variants required for the six experiments:

### Original TF
\[
TF(t,d)=\frac{f(t,d)}{\text{total extracted features in }d}
\]

### Total-word normalised TF
\[
TF(t,d)=\frac{f(t,d)}{\text{number of words in }d}
\]

### Maximum-frequency normalised TF
\[
TF(t,d)=\frac{f(t,d)}{\max_w f(w,d)}
\]

In [11]:
# Cell 11: Compute the three TF variants for one sentence.

def compute_tf(sentence, method):
    features = extract_features(sentence)

    counts = {}

    for feature in features:
        counts[feature] = counts.get(feature, 0) + 1

    if method == "original":
        denominator = len(features)

    elif method == "total_words":
        denominator = len(sentence.split())

    elif method == "max_frequency":
        denominator = max(counts.values()) if counts else 1

    else:
        raise ValueError("Unknown TF method.")

    if denominator == 0:
        return {}

    return {
        feature: count / denominator
        for feature, count in counts.items()
    }

## 8. TF-IDF vector construction

A sparse dictionary representation is used instead of a dense matrix because the n-gram vocabulary can be very large.

In [12]:
# Cell 12: Build a sparse TF-IDF vector for any TF/IDF combination.

def compute_tfidf(sentence, tf_method, idf_method):
    tf = compute_tf(sentence, tf_method)

    if idf_method == "original":
        idf_values = idf_original
    elif idf_method == "normalized":
        idf_values = idf_normalized
    else:
        raise ValueError("Unknown IDF method.")

    vector = {}

    for feature, tf_value in tf.items():
        if feature in vocabulary:
            index = vocabulary[feature]
            vector[index] = tf_value * idf_values[feature]

    return vector

# MODEL 1
## Model 1 — Original TF + Original IDF

**Baseline:** both TF and IDF use the unnormalised formulations from Assignment 1.

In [13]:
# Model 1 - Cell 13: Encode the 23 language labels as class indices.

label_to_index = {}
index_to_label = {}

for i, language in enumerate(languages):
    label_to_index[language] = i
    index_to_label[i] = language

print("Number of classes:", len(languages))

Number of classes: 23


In [14]:
# Model 1 - Cell 14: Convert training, validation, and test sentences to sparse TF-IDF vectors.

def prepare_features(data, tf_method, idf_method):
    X = []
    y = []

    for sentence, label in data:
        X.append(compute_tfidf(sentence, tf_method, idf_method))
        y.append(label_to_index[label])

    return X, y

X_train_1, y_train_1 = prepare_features(train, "original", "original")
X_val_1, y_val_1 = prepare_features(validation, "original", "original")
X_test_1, y_test_1 = prepare_features(test, "original", "original")

print("Train:", len(X_train_1))
print("Validation:", len(X_val_1))
print("Test:", len(X_test_1))

Train: 18400
Validation: 2300
Test: 2300


### Logistic Regression implementation used by all six models

The classifier is multiclass softmax Logistic Regression trained with stochastic gradient descent. The implementation is shared so the only experimental change is the TF-IDF representation.

In [15]:
# Shared model code: softmax function for multiclass Logistic Regression.

def softmax(scores):
    maximum = max(scores)
    exponentials = [math.exp(s - maximum) for s in scores]
    total = sum(exponentials)
    return [value / total for value in exponentials]

In [16]:
# Shared model code: train sparse multiclass Logistic Regression with SGD.

def train_logistic_regression(X, y, num_features, num_classes,
                              epochs=5, learning_rate=0.05, seed=42):
    rng = np.random.default_rng(seed)

    weights = rng.normal(
        0.0,
        0.001,
        size=(num_classes, num_features)
    )

    bias = np.zeros(num_classes)

    order = list(range(len(X)))

    for epoch in range(epochs):
        rng.shuffle(order)
        total_loss = 0.0

        for sample_index in order:
            vector = X[sample_index]
            true_class = y[sample_index]

            scores = []

            for c in range(num_classes):
                score = bias[c]

                for feature_index, value in vector.items():
                    score += weights[c, feature_index] * value

                scores.append(score)

            probabilities = softmax(scores)

            total_loss += -math.log(
                probabilities[true_class] + 1e-15
            )

            for c in range(num_classes):
                error = probabilities[c] - (
                    1.0 if c == true_class else 0.0
                )

                for feature_index, value in vector.items():
                    weights[c, feature_index] -= (
                        learning_rate * error * value
                    )

                bias[c] -= learning_rate * error

        average_loss = total_loss / len(X)

        print(
            f"Epoch {epoch + 1}/{epochs} - "
            f"Loss: {average_loss:.6f}"
        )

    return weights, bias

In [17]:
# Shared model code: predict labels using sparse TF-IDF vectors.

def predict_one(vector, weights, bias):
    scores = []

    for c in range(weights.shape[0]):
        score = bias[c]

        for feature_index, value in vector.items():
            score += weights[c, feature_index] * value

        scores.append(score)

    return max(range(len(scores)), key=lambda i: scores[i])


def predict_dataset(X, weights, bias):
    return [
        predict_one(vector, weights, bias)
        for vector in X
    ]

In [18]:
# Shared evaluation code: compute Macro-F1 from scratch.

def macro_f1_score(y_true, y_pred, num_classes):
    f1_scores = []

    for c in range(num_classes):
        tp = 0
        fp = 0
        fn = 0

        for actual, predicted in zip(y_true, y_pred):
            if actual == c and predicted == c:
                tp += 1
            elif actual != c and predicted == c:
                fp += 1
            elif actual == c and predicted != c:
                fn += 1

        precision_denominator = tp + fp
        recall_denominator = tp + fn

        precision = (
            tp / precision_denominator
            if precision_denominator > 0 else 0.0
        )

        recall = (
            tp / recall_denominator
            if recall_denominator > 0 else 0.0
        )

        if precision + recall == 0:
            f1 = 0.0
        else:
            f1 = (
                2 * precision * recall
                / (precision + recall)
            )

        f1_scores.append(f1)

    return sum(f1_scores) / num_classes


def accuracy_score(y_true, y_pred):
    correct = sum(
        actual == predicted
        for actual, predicted in zip(y_true, y_pred)
    )
    return correct / len(y_true)

In [19]:
# Model 1 - Cell 15: Train the baseline Logistic Regression model.

start_time = time.time()

weights_1, bias_1 = train_logistic_regression(
    X_train_1,
    y_train_1,
    num_features=len(vocabulary),
    num_classes=len(languages),
    epochs=5,
    learning_rate=0.05,
    seed=42
)

pred_1 = predict_dataset(X_test_1, weights_1, bias_1)

model1_f1 = macro_f1_score(
    y_test_1,
    pred_1,
    len(languages)
)

model1_accuracy = accuracy_score(y_test_1, pred_1)

model1_time = time.time() - start_time

print("Model 1 Test Accuracy:", model1_accuracy)
print("Model 1 Test Macro-F1:", model1_f1)
print("Training/evaluation time:", round(model1_time, 2), "seconds")

Epoch 1/5 - Loss: 2.887335
Epoch 2/5 - Loss: 2.455583
Epoch 3/5 - Loss: 2.118295
Epoch 4/5 - Loss: 1.844717
Epoch 5/5 - Loss: 1.621604
Model 1 Test Accuracy: 0.8782608695652174
Model 1 Test Macro-F1: 0.8806224623077661
Training/evaluation time: 558.53 seconds


# MODEL 2
## Model 2 — Total-Word Normalised TF + Original IDF

Only the TF calculation changes. IDF remains the original Assignment 1 IDF.

In [20]:
# Model 2 - Cell 16: Prepare TF-IDF vectors using total-word normalised TF.

X_train_2, y_train_2 = prepare_features(train, "total_words", "original")
X_val_2, y_val_2 = prepare_features(validation, "total_words", "original")
X_test_2, y_test_2 = prepare_features(test, "total_words", "original")

print("Model 2 feature preparation complete.")

Model 2 feature preparation complete.


In [21]:
# Model 2 - Cell 17: Train and evaluate Logistic Regression.

start_time = time.time()

weights_2, bias_2 = train_logistic_regression(
    X_train_2, y_train_2,
    len(vocabulary), len(languages),
    epochs=5,
    learning_rate=0.05,
    seed=42
)

pred_2 = predict_dataset(X_test_2, weights_2, bias_2)

model2_f1 = macro_f1_score(y_test_2, pred_2, len(languages))
model2_accuracy = accuracy_score(y_test_2, pred_2)
model2_time = time.time() - start_time

print("Model 2 Test Accuracy:", model2_accuracy)
print("Model 2 Test Macro-F1:", model2_f1)
print("Training/evaluation time:", round(model2_time, 2), "seconds")

Epoch 1/5 - Loss: 0.532369
Epoch 2/5 - Loss: 0.195230
Epoch 3/5 - Loss: 0.181910
Epoch 4/5 - Loss: 0.185997
Epoch 5/5 - Loss: 0.185100
Model 2 Test Accuracy: 0.9108695652173913
Model 2 Test Macro-F1: 0.9160500631794091
Training/evaluation time: 517.98 seconds


# MODEL 3
## Model 3 — Maximum-Frequency Normalised TF + Original IDF

TF is divided by the frequency of the most frequent feature in the sentence. IDF remains original.

In [22]:
# Model 3 - Cell 18: Prepare TF-IDF vectors using maximum-frequency normalised TF.

X_train_3, y_train_3 = prepare_features(train, "max_frequency", "original")
X_val_3, y_val_3 = prepare_features(validation, "max_frequency", "original")
X_test_3, y_test_3 = prepare_features(test, "max_frequency", "original")

print("Model 3 feature preparation complete.")

Model 3 feature preparation complete.


In [23]:
# Model 3 - Cell 19: Train and evaluate Logistic Regression.

start_time = time.time()

weights_3, bias_3 = train_logistic_regression(
    X_train_3, y_train_3,
    len(vocabulary), len(languages),
    epochs=5,
    learning_rate=0.05,
    seed=42
)

pred_3 = predict_dataset(X_test_3, weights_3, bias_3)

model3_f1 = macro_f1_score(y_test_3, pred_3, len(languages))
model3_accuracy = accuracy_score(y_test_3, pred_3)
model3_time = time.time() - start_time

print("Model 3 Test Accuracy:", model3_accuracy)
print("Model 3 Test Macro-F1:", model3_f1)
print("Training/evaluation time:", round(model3_time, 2), "seconds")

Epoch 1/5 - Loss: 1.193682
Epoch 2/5 - Loss: 0.355669
Epoch 3/5 - Loss: 0.203111
Epoch 4/5 - Loss: 0.190837
Epoch 5/5 - Loss: 0.188757
Model 3 Test Accuracy: 0.8965217391304348
Model 3 Test Macro-F1: 0.9021123056161128
Training/evaluation time: 469.64 seconds


# MODEL 4
## Model 4 — Original TF + Normalised IDF

TF remains unchanged. Only IDF is replaced by the normalised IDF.

In [24]:
# Model 4 - Cell 20: Prepare TF-IDF vectors using original TF and normalised IDF.

X_train_4, y_train_4 = prepare_features(train, "original", "normalized")
X_val_4, y_val_4 = prepare_features(validation, "original", "normalized")
X_test_4, y_test_4 = prepare_features(test, "original", "normalized")

print("Model 4 feature preparation complete.")

Model 4 feature preparation complete.


In [25]:
# Model 4 - Cell 21: Train and evaluate Logistic Regression.

start_time = time.time()

weights_4, bias_4 = train_logistic_regression(
    X_train_4, y_train_4,
    len(vocabulary), len(languages),
    epochs=5,
    learning_rate=0.05,
    seed=42
)

pred_4 = predict_dataset(X_test_4, weights_4, bias_4)

model4_f1 = macro_f1_score(y_test_4, pred_4, len(languages))
model4_accuracy = accuracy_score(y_test_4, pred_4)
model4_time = time.time() - start_time

print("Model 4 Test Accuracy:", model4_accuracy)
print("Model 4 Test Macro-F1:", model4_f1)
print("Training/evaluation time:", round(model4_time, 2), "seconds")

Epoch 1/5 - Loss: 3.082840
Epoch 2/5 - Loss: 2.976027
Epoch 3/5 - Loss: 2.887426
Epoch 4/5 - Loss: 2.808974
Epoch 5/5 - Loss: 2.734251
Model 4 Test Accuracy: 0.5208695652173913
Model 4 Test Macro-F1: 0.5403765537836145
Training/evaluation time: 460.34 seconds


# MODEL 5
## Model 5 — Total-Word Normalised TF + Normalised IDF

Both TF and IDF use their normalised formulations.

In [26]:
# Model 5 - Cell 22: Prepare TF-IDF vectors using total-word normalised TF and normalised IDF.

X_train_5, y_train_5 = prepare_features(train, "total_words", "normalized")
X_val_5, y_val_5 = prepare_features(validation, "total_words", "normalized")
X_test_5, y_test_5 = prepare_features(test, "total_words", "normalized")

print("Model 5 feature preparation complete.")

Model 5 feature preparation complete.


In [27]:
# Model 5 - Cell 23: Train and evaluate Logistic Regression.

start_time = time.time()

weights_5, bias_5 = train_logistic_regression(
    X_train_5, y_train_5,
    len(vocabulary), len(languages),
    epochs=5,
    learning_rate=0.05,
    seed=42
)

pred_5 = predict_dataset(X_test_5, weights_5, bias_5)

model5_f1 = macro_f1_score(y_test_5, pred_5, len(languages))
model5_accuracy = accuracy_score(y_test_5, pred_5)
model5_time = time.time() - start_time

print("Model 5 Test Accuracy:", model5_accuracy)
print("Model 5 Test Macro-F1:", model5_f1)
print("Training/evaluation time:", round(model5_time, 2), "seconds")

Epoch 1/5 - Loss: 0.952551
Epoch 2/5 - Loss: 0.300265
Epoch 3/5 - Loss: 0.228983
Epoch 4/5 - Loss: 0.201734
Epoch 5/5 - Loss: 0.187032
Model 5 Test Accuracy: 0.9182608695652174
Model 5 Test Macro-F1: 0.9227477654210681
Training/evaluation time: 447.37 seconds


# MODEL 6
## Model 6 — Maximum-Frequency Normalised TF + Normalised IDF

Both TF and IDF use their normalised formulations.

In [28]:
# Model 6 - Cell 24: Prepare TF-IDF vectors using maximum-frequency normalised TF and normalised IDF.

X_train_6, y_train_6 = prepare_features(train, "max_frequency", "normalized")
X_val_6, y_val_6 = prepare_features(validation, "max_frequency", "normalized")
X_test_6, y_test_6 = prepare_features(test, "max_frequency", "normalized")

print("Model 6 feature preparation complete.")

Model 6 feature preparation complete.


In [29]:
# Model 6 - Cell 25: Train and evaluate Logistic Regression.

start_time = time.time()

weights_6, bias_6 = train_logistic_regression(
    X_train_6, y_train_6,
    len(vocabulary), len(languages),
    epochs=5,
    learning_rate=0.05,
    seed=42
)

pred_6 = predict_dataset(X_test_6, weights_6, bias_6)

model6_f1 = macro_f1_score(y_test_6, pred_6, len(languages))
model6_accuracy = accuracy_score(y_test_6, pred_6)
model6_time = time.time() - start_time

print("Model 6 Test Accuracy:", model6_accuracy)
print("Model 6 Test Macro-F1:", model6_f1)
print("Training/evaluation time:", round(model6_time, 2), "seconds")

Epoch 1/5 - Loss: 0.495396
Epoch 2/5 - Loss: 0.167006
Epoch 3/5 - Loss: 0.149770
Epoch 4/5 - Loss: 0.145944
Epoch 5/5 - Loss: 0.145504
Model 6 Test Accuracy: 0.91
Model 6 Test Macro-F1: 0.916680930955107
Training/evaluation time: 457.38 seconds


# FINAL RESULTS
## Comparison of All Six TF-IDF Combinations

The final comparison uses the test-set Accuracy and Macro-F1 obtained from the six models.

In [30]:
# Final Results - Cell 26: Create and display the six-model comparison.

results = [
    ("Model 1", "Original TF", "Original IDF", model1_accuracy, model1_f1),
    ("Model 2", "Total-word normalised TF", "Original IDF", model2_accuracy, model2_f1),
    ("Model 3", "Maximum-frequency normalised TF", "Original IDF", model3_accuracy, model3_f1),
    ("Model 4", "Original TF", "Normalised IDF", model4_accuracy, model4_f1),
    ("Model 5", "Total-word normalised TF", "Normalised IDF", model5_accuracy, model5_f1),
    ("Model 6", "Maximum-frequency normalised TF", "Normalised IDF", model6_accuracy, model6_f1),
]

print(f"{'Model':<10} {'TF':<35} {'IDF':<20} {'Accuracy':>12} {'Macro-F1':>12}")
print("-" * 95)

for model, tf_name, idf_name, accuracy, f1 in results:
    print(
        f"{model:<10} "
        f"{tf_name:<35} "
        f"{idf_name:<20} "
        f"{accuracy:>12.4f} "
        f"{f1:>12.4f}"
    )

Model      TF                                  IDF                      Accuracy     Macro-F1
-----------------------------------------------------------------------------------------------
Model 1    Original TF                         Original IDF               0.8783       0.8806
Model 2    Total-word normalised TF            Original IDF               0.9109       0.9161
Model 3    Maximum-frequency normalised TF     Original IDF               0.8965       0.9021
Model 4    Original TF                         Normalised IDF             0.5209       0.5404
Model 5    Total-word normalised TF            Normalised IDF             0.9183       0.9227
Model 6    Maximum-frequency normalised TF     Normalised IDF             0.9100       0.9167


In [31]:
# Final Results - Cell 27: Identify the best model by Macro-F1.

best_result = max(results, key=lambda row: row[4])

print("Best model based on Test Macro-F1:")
print()
print("Model   :", best_result[0])
print("TF      :", best_result[1])
print("IDF     :", best_result[2])
print("Accuracy:", round(best_result[3], 4))
print("Macro-F1 :", round(best_result[4], 4))

Best model based on Test Macro-F1:

Model   : Model 5
TF      : Total-word normalised TF
IDF     : Normalised IDF
Accuracy: 0.9183
Macro-F1 : 0.9227


# CONCLUSION

The six TF-IDF configurations are compared using the same multilingual dataset, feature representation, train/test split, Logistic Regression implementation, and Macro-F1 evaluation.

The best configuration is selected based on the highest test-set Macro-F1.

**Note:** The numerical results below are produced by running the notebook and should not be filled in manually before execution.

## Reproducibility Notes

- Dataset: IndicCorpV2
- Languages: 23
- Samples: 1000 sentences per language
- Total samples: 23,000
- Split: 80% training, 10% validation, 10% test
- Features: word unigrams, word bigrams, character 2-, 3-, and 4-grams
- Classifier: multiclass Logistic Regression implemented from scratch
- Evaluation: Accuracy and Macro-F1 implemented from scratch
- Random seed: 42
- No scikit-learn is used.